# 随机森林回归：多棵树取平均，稳定且强大
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：随机森林_回归.py | 核心功能：Bagging 集成、OOB 评估、特征重要性、参数影响分析

## 概述

随机森林回归 = Bagging + 决策树回归。每棵树用不同的 bootstrap 样本和随机特征子集训练，最终预测取所有树的平均值。单棵树过拟合的高方差问题被"平均"大幅缓解。

脚本演示了完整的随机森林回归工作流：模型训练、OOB 评估、特征重要性可视化、n_estimators 和 max_features 的影响分析，以及与单棵决策树的对比。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 数学原理

### 1. Bagging 的方差缩减原理

**代码对应**：`RandomForestRegressor(n_estimators=100, max_features="sqrt")` 训练随机森林。

随机森林基于 **Bagging**（Bootstrap Aggregating）思想。对 $B$ 棵树的预测取平均：

$$\hat{f}_{\text{RF}}(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^{B} \hat{f}_b(\mathbf{x})$$

设单棵树的方差为 $\sigma^2$，树间相关系数为 $\rho$，则随机森林的方差为：

$$\text{Var}[\hat{f}_{\text{RF}}] = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

当 $B \to \infty$ 时，方差趋近 $\rho\sigma^2$。因此**降低树间相关性 $\rho$** 比增加树的数量 $B$ 更重要。

### 2. 随机特征选择：降低相关性

**代码对应**：`max_features="sqrt"` 限制每次分裂只考虑 $\lfloor\sqrt{p}\rfloor$ 个随机特征。

随机森林在 Bagging 基础上增加**随机特征选择**：每次分裂时，不考虑所有 $p$ 个特征，而是随机选择 $m \leq p$ 个特征（`max_features`），然后从中找最优分裂。

这强制不同树使用不同特征，降低了树间相关性 $\rho$：

| `max_features` | $\rho$ | 树间多样性 | 单棵树质量 | 集成效果 |
|----------------|--------|-----------|-----------|---------|
| $p$（所有特征） | 高 | 低 | 最优 | 较差 |
| $\sqrt{p}$ | 中 | 中 | 中等 | 最佳（分类） |
| $p/3$ | 低 | 高 | 较差 | 最佳（回归，sklearn 默认） |

### 3. OOB（Out-of-Bag）误差估计

**代码对应**：`oob_score=True` 启用 OOB 评分。

每棵树用 Bootstrap 采样训练，约 36.8% 的样本未被抽中（OOB）。样本 $i$ 的 OOB 预测为：

$$\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|}\sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i)$$

其中 $\mathcal{B}_i = \{b : \mathbf{x}_i \notin D_b^{\text{boot}}\}$ 为不包含样本 $i$ 的树的集合。

OOB 误差 $R^2_{\text{OOB}}$ 是泛化误差的无偏估计，等价于无穷折交叉验证，但**完全免费**——不需要单独的验证集。

### 4. 特征重要性计算

**代码对应**：`rf.feature_importances_` 返回特征重要性。

随机森林的特征重要性基于**不纯度减少**（Impurity Decrease）：

$$\text{Imp}(j) = \frac{1}{B}\sum_{b=1}^{B}\sum_{t \in T_b: \text{splits on } j} \frac{n_t}{N}\Delta\text{MSE}_t$$

另一种更可靠的方法是**置换重要性**（Permutation Importance）：随机打乱特征 $j$ 的值，观察 OOB 误差增加多少。增加越多，特征越重要。

### 5. 偏差-方差分解

随机森林相比单棵决策树：

- **偏差**：略有增加（随机特征选择降低了单棵树质量）
- **方差**：大幅降低（平均化 + 随机性降低相关性）

**代码对应**：代码中对比了单棵决策树和随机森林——单棵树训练 $R^2 = 1.0$ 但测试 $R^2$ 较低（过拟合），随机森林训练 $R^2$ 更合理但测试 $R^2$ 显著更高。

$$\text{MSE}_{\text{RF}} = \text{Bias}_{\text{RF}}^2 + \text{Var}_{\text{RF}} + \sigma^2$$

通常 $\text{Bias}_{\text{RF}} \approx \text{Bias}_{\text{tree}}$（略有增加），但 $\text{Var}_{\text{RF}} \ll \text{Var}_{\text{tree}}$（大幅减少）。

### 6. 超参数影响

**代码对应**：代码中分别对比了 `n_estimators` 和 `max_features` 的影响。

- `n_estimators`：树的数量。更多树 $\Rightarrow$ 方差更小（收敛到 $\rho\sigma^2$），但计算量线性增长。通常 100~500 足够
- `max_features`：每棵树的随机特征数。越小 $\Rightarrow$ 树间相关性越低，但单棵树质量下降
- `max_depth`：控制单棵树复杂度。无限制时偏差最低但单棵树方差高

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_regression(
    n_samples=500, n_features=15, n_informative=8,
    noise=15, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 基础随机森林回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100, max_features="sqrt",
    oob_score=True, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

print("=== 随机森林回归 ===")
print(f"训练 R²: {rf.score(X_train, y_train):.4f}")
print(f"测试 R²: {rf.score(X_test, y_test):.4f}")
print(f"OOB R²: {rf.oob_score_:.4f}")

y_pred = rf.predict(X_test)
print(f"MSE:  {mean_squared_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")

### 3. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print("\n=== 特征重要性 ===")
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
for rank, idx in enumerate(indices, 1):
    bar = "█" * int(importances[idx] * 100)
# --- 输出结果 ---
    print(f"  {rank:>2}. 特征{idx:>2}: {importances[idx]:.4f} {bar}")

### 4. n_estimators 影响

运行 4. n_estimators 影响 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== n_estimators 对比 ===")
for n in [10, 50, 100, 200, 500]:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    oob_rf = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
# --- 训练模型 ---
    oob_rf.fit(X_train, y_train)
    print(f"  n={n:>4}: 测试R²={rf_n.score(X_test, y_test):.4f}, OOB={oob_rf.oob_score_:.4f}")

### 5. max_features 影响

运行 5. max_features 影响 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== max_features 对比 ===")
for mf in ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0]:
    rf_m = RandomForestRegressor(n_estimators=100, max_features=mf, random_state=42, n_jobs=-1)
    rf_m.fit(X_train, y_train)
    test_r2 = rf_m.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  max_features={str(mf):>6}: 测试R²={test_r2:.4f}")

### 6. 单棵树 vs 随机森林

运行 6. 单棵树 vs 随机森林 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 单棵树 vs 随机森林 ===")
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
print(f"决策树:    训练R²={dt.score(X_train, y_train):.4f}, 测试R²={dt.score(X_test, y_test):.4f}")
# --- 输出结果 ---
print(f"随机森林:  训练R²={rf.score(X_train, y_train):.4f}, 测试R²={rf.score(X_test, y_test):.4f}")
print("随机森林通过集成显著降低过拟合（训练R²从1.0降到更合理的值，测试R²大幅提升）")

print("\n=== 随机森林回归要点 ===")
print("- 每棵树用 bootstrap 采样 + 随机特征子集训练")
print("- 最终预测 = 所有树预测的平均值")
print("- OOB 评分提供免费的验证，无需单独划分验证集")
print("- 通常 n_estimators=100~500 即可，更多收益递减")
# --- 输出结果 ---
print("- 不需要特征缩放，能捕捉非线性关系和交互效应")

## 关键代码解释

### OOB 评分——免费的交叉验证

```python
rf = RandomForestRegressor(n_estimators=100, oob_score=True, n_jobs=-1)
```

每棵 bootstrap 树约用 63.2% 的训练样本，剩余 36.8% 的袋外样本可用于评估。OOB 评分近似交叉验证，无需额外划分验证集。

### 特征重要性

```python
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
```

基于每个特征在所有树中带来的 MSE 降低总量排序。

### max_features 调节

```python
for mf in ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0]:
```

回归任务默认 max_features=n_features（即所有特征），但实际中用 sqrt 或 log2 有时效果更好（增加树间的差异性）。

## 使用示例

```python
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=200, oob_score=True, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"OOB R²: {rf.oob_score_:.4f}")
```

## 注意事项

1. **不需要特征缩放**
2. **外推能力差**：与决策树一样，不能预测训练范围之外的值
3. **n_jobs=-1 并行化很重要**
4. **大数据集训练可能慢**：考虑 HistGradientBoostingRegressor

## 总结与延伸

以上代码展示了 **随机森林 回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **ExtraTreesRegressor**：更极端的随机性（分裂阈值也随机），更快但方差更大
- **条件重要性**：考虑特征交互的特征重要性度量
- **与 XGBoost 对比**：随机森林是 Bagging（并行），XGBoost 是 Boosting（串行），通常后者精度更高
